In [ ]:
import subprocess, sys
def pip(*pkgs):
    subprocess.check_call([sys.executable,"-m","pip","install","-q",*pkgs])
pip("portpy","cvxpy","clarabel","scipy","matplotlib",
    "numpy","h5py","huggingface_hub","pandas","pydicom")

# ══════════════════════════════════════════════════════════════════════════════
# 1. IMPORTS
# ══════════════════════════════════════════════════════════════════════════════
import os, json, warnings, time
import h5py
import numpy as np
import scipy.sparse as sp
from scipy.sparse import coo_matrix, hstack
from scipy.sparse import linalg as splinalg
from scipy.interpolate import RegularGridInterpolator
import cvxpy as cp
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import pandas as pd
import pydicom
from huggingface_hub import snapshot_download
from IPython.display import display, Image as IPImage

warnings.filterwarnings("ignore")

# ── Plot style constants ──────────────────────────────────────────────────────
DARK    = "#0e1117"
GRID_C  = "#2a2a2a"
SPINE_C = "#444444"
PAL     = ["#aaaaaa", "#00c8ff", "#a8ff78", "#ffd166"]   # ref, λ=0, λ=0.001, λ=0.003
SC      = {"PTV":"#00c8ff","Esoph":"#ffd166","Cord":"#ff9f43","Lung":"#a29bfe"}
LS      = ["-","--","-.",":"]

OUTPUT_DIR = "/mnt/user-data/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def save_and_show(fig, filename):
    """Save figure to outputs directory and display inline in notebook."""
    path = os.path.join(OUTPUT_DIR, filename)
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=DARK)
    plt.close(fig)
    print(f"  Saved → {path}")
    try:
        display(IPImage(filename=path))
    except Exception:
        pass   # not in a notebook environment — that's fine

def style_ax(ax, title=None, xlabel=None, ylabel=None):
    """Apply consistent dark theme to an axis."""
    ax.set_facecolor(DARK)
    ax.tick_params(colors="white")
    ax.grid(color=GRID_C, lw=0.5)
    for sp in ax.spines.values():
        sp.set_edgecolor(SPINE_C)
    if title:  ax.set_title(title,  color="white", fontsize=10, pad=8)
    if xlabel: ax.set_xlabel(xlabel, color="white", fontsize=9)
    if ylabel: ax.set_ylabel(ylabel, color="white", fontsize=9)

# ══════════════════════════════════════════════════════════════════════════════
# 2. CLINICAL CONSTANTS
# ══════════════════════════════════════════════════════════════════════════════
D_PTV   = 60.0;  D_ESOPH = 45.0;  D_CORD = 30.0;  D_LUNG = 20.0
N_FRACS = 30;    W1_0    = 10.0;  W2_0   =  2.0

# ══════════════════════════════════════════════════════════════════════════════
# 3. DOWNLOAD
# ══════════════════════════════════════════════════════════════════════════════
print("="*68)
print("STEP 1 — Download PortPy Lung_Patient_3")
print("="*68)
snapshot_download(repo_id="PortPy-Project/PortPy_Dataset",
                  repo_type="dataset",
                  allow_patterns="data/Lung_Patient_3/*",
                  local_dir="./hf_data")
DATA_DIR  = "./hf_data/data/Lung_Patient_3"
DICOM_DIR = os.path.join(DATA_DIR,"DicomFiles")
print(f"Top-level: {sorted(os.listdir(DATA_DIR))}")
print(f"DICOM:     {sorted(os.listdir(DICOM_DIR)) if os.path.isdir(DICOM_DIR) else 'missing'}\n")

# ══════════════════════════════════════════════════════════════════════════════
# 4. BUILD INFLUENCE MATRIX + STRUCTURES
# ══════════════════════════════════════════════════════════════════════════════
print("="*68)
print("STEP 2 — Build influence matrix and structure masks")
print("="*68)

beam_dir   = os.path.join(DATA_DIR,"Beams")
beam_files = sorted([os.path.join(beam_dir,f) for f in os.listdir(beam_dir)
                     if f.endswith("_Data.h5")])

with h5py.File(beam_files[0],"r") as f:
    n_voxels = f["inf_matrix_full"].shape[0]

A_blocks, beamlets_per_beam = [], []
for bf in beam_files:
    with h5py.File(bf,"r") as f:
        coo = f["inf_matrix_sparse"][:]
        nc  = f["inf_matrix_full"].shape[1]
        A_blocks.append(coo_matrix(
            (coo[:,2].astype(np.float32),
             (coo[:,0].astype(np.int32), coo[:,1].astype(np.int32))),
            shape=(n_voxels, nc)).tocsr())
        beamlets_per_beam.append(nc)

A_full = hstack(A_blocks, format="csr")

with h5py.File(os.path.join(DATA_DIR,"OptimizationVoxels_Data.h5"),"r") as f:
    opt_keys   = list(f.keys())
    ct_to_dose = f["ct_to_dose_voxel_map"][:]
    print(f"OptVoxels keys: {opt_keys}")

ct_meta_file = os.path.join(DATA_DIR,"CT_MetaData.json")
ct_meta      = None
coord_mm     = None

if os.path.exists(ct_meta_file):
    with open(ct_meta_file) as f:
        ct_meta = json.load(f)
    print(f"CT_MetaData keys: {list(ct_meta.keys())}")

    def get(d, *keys, default=None):
        for k in keys:
            if k in d: return d[k]
        return default

    origin  = np.array(get(ct_meta,"origin","Origin","ImagePositionPatient","origin_xyz_mm"), dtype=float)
    spacing = np.array(get(ct_meta,"resolution","Resolution","PixelSpacing","voxel_size","resolution_xyz_mm"), dtype=float)
    ct_size = np.array(get(ct_meta,"size","Size","Dimensions","size_xyz_mm"), dtype=int)

    print(f"  CT origin  : {origin}")
    print(f"  CT spacing : {spacing}")
    print(f"  CT size    : {ct_size}")

    if origin is not None and spacing is not None and ct_size is not None:
        if len(spacing) == 2:
            z_sp = get(ct_meta,"slice_thickness","SliceThickness","z_spacing","spacing_z",default=3.0)
            spacing = np.array([spacing[0], spacing[1], float(z_sp)])

        nx, ny, nz = int(ct_size[0]), int(ct_size[1]), int(ct_size[2])
        valid      = ct_to_dose >= 0
        lin_idx    = ct_to_dose[valid].astype(np.int64)

        for order, shape in [("C",(nz,ny,nx)),("F",(nx,ny,nz)),
                              ("C",(nx,ny,nz)),("C",(ny,nx,nz))]:
            try:
                idxs = np.array(np.unravel_index(lin_idx, shape, order=order))
                xs = origin[0] + idxs[0]*spacing[0]
                ys = origin[1] + idxs[1]*spacing[1]
                zs = origin[2] + idxs[2]*spacing[2]
                if (xs.min() > -1000 and xs.max() < 1000 and
                    ys.min() > -1000 and ys.max() < 1000):
                    coord_mm = np.zeros((n_voxels, 3))
                    coord_mm[valid, 0] = xs
                    coord_mm[valid, 1] = ys
                    coord_mm[valid, 2] = zs
                    break
            except Exception:
                continue

with h5py.File(os.path.join(DATA_DIR,"StructureSet_Data.h5"),"r") as f:
    mask_ptv   = f["PTV"][:]
    mask_esoph = f["ESOPHAGUS"][:]
    mask_cord  = f["CORD"][:]
    mask_lung  = f["LUNG_L"][:]

def vox_idx(mask):
    both = mask.astype(bool) & (ct_to_dose >= 0)
    return ct_to_dose[both].astype(np.int32)

ptv_v=vox_idx(mask_ptv); esoph_v=vox_idx(mask_esoph)
cord_v=vox_idx(mask_cord); lung_v=vox_idx(mask_lung)

with open(os.path.join(DATA_DIR,"PlannerBeams.json")) as f:
    planner  = json.load(f)
beam_ids = planner if isinstance(planner,list) else list(planner.values())[0]

beam_col_start = np.cumsum([0]+beamlets_per_beam)
plan_cols      = np.concatenate([
    np.arange(beam_col_start[b], beam_col_start[b+1]) for b in beam_ids])

def sub_A(vox):
    return A_full[vox,:][:,plan_cols].astype(np.float64).tocsr()

A_ptv=sub_A(ptv_v); A_esoph=sub_A(esoph_v)
A_cord=sub_A(cord_v); A_lung=sub_A(lung_v)

plan_bpb   = [beamlets_per_beam[b] for b in beam_ids]
nb         = len(beam_ids); n_beamlets=sum(plan_bpb); max_bpb=max(plan_bpb)
nP=A_ptv.shape[0]; nE=A_esoph.shape[0]; nC=A_cord.shape[0]; nL=A_lung.shape[0]

print(f"\nPlanner beams: {nb}  bpb={plan_bpb}  total={n_beamlets}")
print(f"PTV={nP} Esoph={nE} Cord={nC} Lung={nL}\n")

def dose(A,x): return np.asarray(A @ x).ravel()

def to_X(xv):
    starts=np.cumsum([0]+plan_bpb[:-1]); X=np.zeros((nb,max_bpb))
    for i,(s,sz) in enumerate(zip(starts,plan_bpb)): X[i,:sz]=np.maximum(xv[s:s+sz],0)
    return X

def svd_m(xv):
    X=to_X(xv)
    try: svs=np.linalg.svd(X,compute_uv=False)
    except: svs=np.array([np.linalg.norm(X,"fro")])
    nuc=float(svs.sum()); rank=float((svs.sum()**2)/((svs**2).sum()+1e-12))
    return nuc,rank

def print_plan(label,xv):
    xv=np.maximum(xv,0)
    dp=dose(A_ptv,xv); de=dose(A_esoph,xv)
    dc=dose(A_cord,xv); dl=dose(A_lung,xv)
    nuc,rnk=svd_m(xv)
    p=["✓" if np.percentile(dp,5)>=57 else "✗",
       "✓" if de.max()<=45 else "✗",
       "✓" if dc.max()<=30 else "✗",
       "✓" if dl.mean()<=20 else "✗"]
    print(f"  [{label}]")
    print(f"    PTV D95={np.percentile(dp,5):.2f}{p[0]} D05={np.percentile(dp,95):.2f} Dmean={dp.mean():.2f} Gy")
    print(f"    Esoph Dmax={de.max():.2f}{p[1]}  Cord Dmax={dc.max():.2f}{p[2]}  Lung Dmean={dl.mean():.2f}{p[3]} Gy")
    print(f"    NucNorm={nuc:.1f}  EffRank={rnk:.3f}  MU={xv.sum():.1f}  nnz={np.sum(xv>1e-3)}/{n_beamlets}")

# ══════════════════════════════════════════════════════════════════════════════
# 5. PHASE 1 — REFERENCE PLAN
# ══════════════════════════════════════════════════════════════════════════════
print("="*68)
print("PHASE 1 — ECHO Reference Plan")
print("="*68)

x_ref = None; ref_source = "none"

def find_dicom(directory, modality):
    if not os.path.isdir(directory): return None
    for fname in sorted(os.listdir(directory)):
        if not fname.lower().endswith(".dcm"): continue
        try:
            ds=pydicom.dcmread(os.path.join(directory,fname),stop_before_pixels=True)
            if ds.Modality==modality: return os.path.join(directory,fname)
        except: continue
    return None

rtdose_file = find_dicom(DICOM_DIR,"RTDOSE")
if rtdose_file: print(f"  RTDOSE found: {os.path.basename(rtdose_file)}")
else:           print("  No RTDOSE in DicomFiles/")

if rtdose_file and coord_mm is not None:
    try:
        ds = pydicom.dcmread(rtdose_file)
        dose_grid  = ds.pixel_array.astype(np.float64) * float(ds.DoseGridScaling)
        dose_sum   = str(getattr(ds,"DoseSummationType","PLAN")).upper()
        if dose_sum not in ("PLAN","EFFECTIVE"):
            dose_grid *= N_FRACS

        nz_d,ny_d,nx_d = dose_grid.shape
        print(f"  Dose grid: {dose_grid.shape}  [{dose_grid.min():.1f},{dose_grid.max():.1f}] Gy")

        ipp   = np.array(ds.ImagePositionPatient, dtype=float)
        pixsp = np.array(ds.PixelSpacing, dtype=float)
        z_off = (np.array(ds.GridFrameOffsetVector, dtype=float)
                 if hasattr(ds,"GridFrameOffsetVector")
                 else np.arange(nz_d)*float(getattr(ds,"SliceThickness",3.0)))

        xs_d=np.sort(ipp[0]+np.arange(nx_d)*pixsp[1])
        ys_d=np.sort(ipp[1]+np.arange(ny_d)*pixsp[0])
        zs_d=ipp[2]+z_off; zs_d=np.sort(zs_d)

        rgi = RegularGridInterpolator((zs_d,ys_d,xs_d),dose_grid,
                                      method="linear",bounds_error=False,fill_value=0.0)
        query    = coord_mm[:,[2,1,0]]
        d_ref_f  = rgi(query)

        ptv_d95 = float(np.percentile(d_ref_f[ptv_v],5))
        print(f"\n  RTDOSE interpolated:  PTV D95={ptv_d95:.2f} Gy  "
              f"Esoph Dmax={d_ref_f[esoph_v].max():.2f} Gy  "
              f"Lung Dmean={d_ref_f[lung_v].mean():.2f} Gy")

        if ptv_d95 < 30.0:
            print("  WARNING: PTV D95 too low — using LP fallback")
        else:
            all_vox  = np.concatenate([ptv_v,esoph_v,cord_v,lung_v])
            A_all    = A_full[all_vox,:][:,plan_cols].astype(np.float64).tocsr()
            d_all    = d_ref_f[all_vox]
            res = splinalg.lsqr(A_all, d_all, damp=1e-2, iter_lim=2000, show=False)
            x_ref = np.maximum(res[0], 0.0)
            if float(np.percentile(dose(A_ptv,x_ref),5)) < 40.0:
                x_ref = None
            else:
                ref_source = "RTDOSE DICOM → LSQR back-projection"
                print_plan("RTDOSE reference", x_ref)
    except Exception as ex:
        x_ref = None

if x_ref is None:
    print("\n  Solving baseline LP (w1=10, w2=2) as reference x* …")
    xv = cp.Variable(n_beamlets, nonneg=True)
    prob = cp.Problem(
        cp.Minimize(
            (W1_0/nP)*cp.sum(cp.pos(D_PTV   - A_ptv  @xv)) +
            (W2_0/nE)*cp.sum(cp.pos(A_esoph @xv - D_ESOPH)) +
            (W2_0/nC)*cp.sum(cp.pos(A_cord  @xv - D_CORD))),
        [cp.sum(A_lung@xv)/nL <= D_LUNG])
    try:
        prob.solve(solver=cp.OSQP,eps_abs=1e-4,eps_rel=1e-4,max_iter=20000,verbose=False)
    except Exception:
        prob.solve(solver=cp.SCS,eps=1e-4,max_iters=10000,verbose=False)
    x_ref = np.maximum(xv.value,0.0)
    ref_source = f"Baseline LP (w1={W1_0}, w2={W2_0})"
    print_plan("LP reference", x_ref)

print(f"\n  Reference confirmed  →  {ref_source}")

# ══════════════════════════════════════════════════════════════════════════════
# 6. PHASE 2 — INVERSE OPTIMISATION (KKT SUBGRADIENT QCQP)
# ══════════════════════════════════════════════════════════════════════════════
print("\n"+"="*68)
print("PHASE 2 — Inverse Optimisation  (KKT Subgradient QCQP)")
print("="*68)

d_ptv_r  = dose(A_ptv,  x_ref); d_esoph_r = dose(A_esoph,x_ref)
d_cord_r = dose(A_cord, x_ref); d_lung_r  = dose(A_lung, x_ref)

eps = 1e-6
s1  = np.where(d_ptv_r  < D_PTV  -eps, -1.0/nP, 0.0)
s2  = np.where(d_esoph_r> D_ESOPH+eps,  1.0/nE, 0.0)
s3  = np.where(d_cord_r > D_CORD +eps,  1.0/nC, 0.0)

g1   = np.asarray(A_ptv.T   @ s1).ravel()
gOAR = np.asarray(A_esoph.T @ s2 + A_cord.T @ s3).ravel()
G    = np.column_stack([g1, gOAR])

print(f"\n  ||g1(PTV)||  = {np.linalg.norm(g1):.6f}")
print(f"  ||gOAR||     = {np.linalg.norm(gOAR):.6f}")

active = x_ref > 1e-3
n_act  = active.sum()
print(f"  Active beamlets: {n_act}/{n_beamlets}")

G_act  = G[active,:]
Q_act  = G_act.T  @ G_act   + 1e-8*np.eye(2)
Q_full = G.T      @ G       + 1e-8*np.eye(2)
Q      = 0.7*Q_act + 0.3*Q_full

w_prior = np.array([W1_0,W2_0])/(W1_0+W2_0)
lam_reg = 1e-3

w_var  = cp.Variable(2, nonneg=True)
io_qp  = cp.Problem(
    cp.Minimize(cp.quad_form(w_var, cp.psd_wrap(Q)) +
                lam_reg*cp.sum_squares(w_var - w_prior)),
    [cp.sum(w_var)==1])
io_qp.solve(solver=cp.CLARABEL, verbose=False)

if w_var.value is None:
    print("  CLARABEL failed — using prior weights")
    w_rec = w_prior.copy()
else:
    w_rec = np.maximum(w_var.value,0.0); w_rec /= w_rec.sum()

kkt_all    = float(np.linalg.norm(np.maximum(G       @ w_rec, 0)))
kkt_active = float(np.linalg.norm(np.maximum(G_act   @ w_rec, 0)))

print(f"\n  Recovered weights w*:")
print(f"    w₁* (PTV)  = {w_rec[0]:.6f}   (prior: {w_prior[0]:.4f}  Δ={w_rec[0]-w_prior[0]:+.4f})")
print(f"    w₂* (OAR)  = {w_rec[1]:.6f}   (prior: {w_prior[1]:.4f}  Δ={w_rec[1]-w_prior[1]:+.4f})")
print(f"\n  KKT residual (all beamlets)    = {kkt_all:.6f}")
print(f"  KKT residual (active beamlets) = {kkt_active:.6f}")
print(f"  (values ≈ 0 → x_ref is optimal under w*)")

# ══════════════════════════════════════════════════════════════════════════════
# 7. PHASE 3 — RE-PLAN WITH w* + SPECTRAL REGULARISATION
# ══════════════════════════════════════════════════════════════════════════════
print("\n"+"="*68)
print("PHASE 3 — Re-plan with w* + Frobenius+Group Spectral Reg.")
print(f"  w₁*={w_rec[0]:.4f}  w₂*={w_rec[1]:.4f}   λ ∈ {{0.0, 0.001, 0.005, 0.01, 0.1}}")
print("="*68)

def solve_replan(w1, w2, lam=0.0):
    t0 = time.time()
    x  = cp.Variable(n_beamlets, nonneg=True)
    total = w1+w2
    W1s   = (w1/total)*(W1_0+W2_0)
    W2s   = (w2/total)*(W1_0+W2_0)
    obj = [
        (W1s/nP)*cp.sum(cp.pos(D_PTV   - A_ptv  @x)),
        (W2s/nE)*cp.sum(cp.pos(A_esoph @x - D_ESOPH)),
        (W2s/nC)*cp.sum(cp.pos(A_cord  @x - D_CORD)),
    ]
    if lam > 0:
        starts  = np.cumsum([0]+plan_bpb[:-1])
        rows_cp = []
        for s,sz in zip(starts,plan_bpb):
            seg = x[s:s+sz]
            if sz < max_bpb:
                seg = cp.hstack([seg, cp.Constant(np.zeros(max_bpb-sz))])
            rows_cp.append(seg)
        Xm = cp.vstack(rows_cp)
        obj.append(lam * cp.norm(Xm,"fro"))
        obj.append(lam * cp.sum(cp.norm(Xm, axis=1)))
    prob = cp.Problem(cp.Minimize(cp.sum(obj)),
                      [cp.sum(A_lung@x)/nL <= D_LUNG])
    try:
        if lam==0:
            prob.solve(solver=cp.OSQP,eps_abs=1e-4,eps_rel=1e-4,
                       max_iter=20000,verbose=False); slv="OSQP"
        else:
            prob.solve(solver=cp.SCS,eps=1e-4,max_iters=10000,verbose=False); slv="SCS"
    except Exception:
        prob.solve(solver=cp.SCS,eps=1e-4,max_iters=10000,verbose=False); slv="SCS(fb)"
    elapsed = time.time()-t0
    xv = np.maximum(x.value,0.0) if x.value is not None else None
    F1 = float(np.mean(np.maximum(D_PTV-A_ptv@xv,0)))     if xv is not None else np.nan
    F2 = float(np.mean(np.maximum(A_esoph@xv-D_ESOPH,0)) +
               np.mean(np.maximum(A_cord @xv-D_CORD, 0))) if xv is not None else np.nan
    return dict(x=xv,lam=lam,status=prob.status,solver=slv,F1=F1,F2=F2,time=elapsed)

REPLANS = {}
lambda_values = [0.0, 0.001, 0.005, 0.01, 0.1]

for lam in lambda_values:
    tag = "IO λ=0 (no reg)" if lam==0 else f"IO λ={lam} (Fro+Group)"
    print(f"\n  [{tag}]  …", end="", flush=True)
    r = solve_replan(w_rec[0], w_rec[1], lam)
    print(f"  {r['status']} [{r['solver']}] {r['time']:.0f}s  F1={r['F1']:.4f}  F2={r['F2']:.4f}")
    if r["x"] is not None: print_plan(tag, r["x"])
    REPLANS[tag] = r

# ══════════════════════════════════════════════════════════════════════════════
# 8. METRICS
# ══════════════════════════════════════════════════════════════════════════════
def metrics(xv, label):
    xv=np.maximum(xv,0)
    dp=dose(A_ptv,xv); de=dose(A_esoph,xv); dc=dose(A_cord,xv); dl=dose(A_lung,xv)
    nuc,rnk=svd_m(xv)
    return {"label":label,
            "D95_ptv"     :round(float(np.percentile(dp, 5)),2),
            "D05_ptv"     :round(float(np.percentile(dp,95)),2),
            "Dmean_ptv"   :round(float(dp.mean()),2),
            "HI"          :round(float((np.percentile(dp,95)-np.percentile(dp,5))/D_PTV),4),
            "CI_proxy_%"  :round(float(np.mean(dp>=0.95*D_PTV)*100),2),
            "F1_underdose":round(float(np.mean(np.maximum(D_PTV-dp,0))),4),
            "Dmax_esoph"  :round(float(de.max()),2),
            "Dmean_esoph" :round(float(de.mean()),2),
            "Dmax_cord"   :round(float(dc.max()),2),
            "Dmean_cord"  :round(float(dc.mean()),2),
            "Dmean_lung"  :round(float(dl.mean()),2),
            "V20_lung_%"  :round(float(np.mean(dl>=20)*100),2),
            "F2_overdose" :round(float(np.mean(np.maximum(de-D_ESOPH,0))+
                                       np.mean(np.maximum(dc-D_CORD,0))),4),
            "nuc_norm"    :round(nuc,1),
            "eff_rank"    :round(rnk,3),
            "sparsity_%"  :round(float(np.mean(xv<1e-3)*100),2),
            "total_MU"    :round(float(xv.sum()),1)}

all_plans = {f"Ref ({ref_source[:20]})":x_ref}
for tag,r in REPLANS.items():
    if r["x"] is not None: all_plans[tag]=r["x"]

rows=[metrics(xp,lbl) for lbl,xp in all_plans.items()]
df=pd.DataFrame(rows).set_index("label")

# ══════════════════════════════════════════════════════════════════════════════
# 9. PLOTTING — FIVE SEPARATE PUBLICATION-QUALITY FIGURES
# ══════════════════════════════════════════════════════════════════════════════
bins   = np.linspace(0,80,300)
labels = list(all_plans.keys())
plans  = list(all_plans.values())
strs   = [("PTV",A_ptv,D_PTV),("Esoph",A_esoph,D_ESOPH),
          ("Cord",A_cord,D_CORD),("Lung",A_lung,D_LUNG)]

lv, f1v, f2v, nv, rv = [], [], [], [], []

for lam in lambda_values:
    tag = "IO λ=0 (no reg)" if lam==0 else f"IO λ={lam} (Fro+Group)"
    r = REPLANS.get(tag)
    if r is None or r["x"] is None:
        continue
    lv.append(lam)
    f1v.append(r["F1"])
    f2v.append(r["F2"])
    n, rk = svd_m(r["x"])
    nv.append(n)
    rv.append(rk)

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 1 — Weight Recovery + KKT Residual Certificate
# ─────────────────────────────────────────────────────────────────────────────
print("\n── Generating Figure 1: Weight Recovery + KKT Residual ──")
fig1, (ax_w, ax_k) = plt.subplots(1, 2, figsize=(12, 5), facecolor=DARK)
fig1.suptitle("IO Phase 2: KKT QCQP Weight Recovery",
              color="white", fontsize=13, fontweight="bold", y=1.02)

# Left — weight bar chart
xp_ = np.arange(2)
bars1 = ax_w.bar(xp_-0.20, w_prior, 0.36,
                 label="Prior (manual normalised)", color="#666666", alpha=0.85, zorder=3)
bars2 = ax_w.bar(xp_+0.20, w_rec,   0.36,
                 label="IO Recovered w*",           color="#00c8ff", alpha=0.90, zorder=3)
for xi,(wp,wr) in enumerate(zip(w_prior,w_rec)):
    ax_w.text(xi-0.20, wp+0.006, f"{wp:.4f}", ha="center", color="white",   fontsize=10, fontweight="bold")
    ax_w.text(xi+0.20, wr+0.006, f"{wr:.4f}", ha="center", color="#00c8ff", fontsize=10, fontweight="bold")
    delta = wr - wp
    sign  = "+" if delta>=0 else ""
    ax_w.text(xi, max(wp,wr)+0.025, f"Δ={sign}{delta:.4f}",
              ha="center", color="#ffd166", fontsize=9)
ax_w.set_xticks(xp_)
ax_w.set_xticklabels(["w₁  (PTV weight)", "w₂  (OAR weight)"], color="white", fontsize=11)
ax_w.set_ylim(0, 1.1)
style_ax(ax_w,
         title="Prior vs Recovered Objective Weights",
         ylabel="Weight value (simplex: w₁ + w₂ = 1)")
ax_w.legend(facecolor="#1c1c2e", labelcolor="white", fontsize=9, loc="upper right")

# Right — KKT residual certificate
residuals  = [kkt_all, kkt_active]
res_labels = ["All beamlets\n(n=4420)", f"Active beamlets\n(n={n_act})"]
res_colors = ["#ff9f43", "#a8ff78"]
bars_k = ax_k.barh(res_labels, residuals, color=res_colors, alpha=0.88, height=0.4, zorder=3)
for bar, val in zip(bars_k, residuals):
    ax_k.text(val + max(residuals)*0.02, bar.get_y()+bar.get_height()/2,
              f"{val:.6f}", va="center", color="white", fontsize=11, fontweight="bold")
ax_k.axvline(0.001, color="#ff6b6b", ls="--", lw=1.5, label="0.001 threshold")
ax_k.axvline(0.000, color="#555",    ls=":",  lw=1.0)
ax_k.set_xlim(0, max(residuals)*1.35)
style_ax(ax_k,
         title="KKT Residual  ‖max(G w*, 0)‖\n(≈ 0 → x* is optimal under w*)",
         xlabel="KKT Residual magnitude")
ax_k.tick_params(axis="y", labelcolor="white", labelsize=10)
ax_k.legend(facecolor="#1c1c2e", labelcolor="white", fontsize=9)
ax_k.text(0.98, 0.08,
          f"Interpretation:\nResidual < 0.001 confirms\nx* is optimal under w*",
          transform=ax_k.transAxes, ha="right", va="bottom",
          color="#a8ff78", fontsize=9,
          bbox=dict(boxstyle="round,pad=0.4", facecolor="#1c1c2e", alpha=0.8))

plt.tight_layout(pad=2.0)
plt.subplots_adjust(top=0.90, hspace=0.35, wspace=0.30)
save_and_show(fig1, "fig1_weight_recovery_kkt.png")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 2 — KKT Geometry (subgradient space)
# ─────────────────────────────────────────────────────────────────────────────
print("\n── Generating Figure 2: KKT Geometry ──")
fig2, axes2 = plt.subplots(1, 2, figsize=(13, 5.5), facecolor=DARK)
fig2.suptitle("IO Phase 2: KKT Geometry in Subgradient Space",
              color="white", fontsize=13, fontweight="bold", y=1.02)

# Left — full scatter (all active beamlets)
ax_geo = axes2[0]
ga = G[active,:]
sc = ax_geo.scatter(ga[:,0], ga[:,1], s=4, alpha=0.25, color="#a8ff78",
                    label=f"Active beamlets (n={n_act})", zorder=2)
# KKT feasibility region: beamlets where G w* ≥ 0 (satisfied) vs < 0 (violated)
kkt_vals  = G[active,:] @ w_rec
satisfied = kkt_vals >= 0
ax_geo.scatter(ga[satisfied,0],  ga[satisfied,1],  s=4, alpha=0.3,
               color="#00c8ff", label=f"KKT satisfied ({satisfied.sum()})", zorder=3)
ax_geo.scatter(ga[~satisfied,0], ga[~satisfied,1], s=4, alpha=0.3,
               color="#ff6b6b", label=f"KKT violated  ({(~satisfied).sum()})",  zorder=3)
# Draw w* direction arrow
# Normalize arrow direction (VERY IMPORTANT)
direction = np.array([w_rec[0], w_rec[1]])
direction = direction / (np.linalg.norm(direction) + 1e-12)

# Scale based on axis limits (NOT raw data)
xlim = ax_geo.get_xlim()
ylim = ax_geo.get_ylim()

scale_arrow = 0.25 * max(abs(xlim[1]-xlim[0]), abs(ylim[1]-ylim[0]))

arrow_end = direction * scale_arrow

# Draw arrow
ax_geo.annotate(
    "",
    xy=(arrow_end[0], arrow_end[1]),
    xytext=(0, 0),
    arrowprops=dict(arrowstyle="->", color="#ffd166", lw=2)
)

# Place text slightly offset from arrow tip
ax_geo.text(
    arrow_end[0]*1.1,
    arrow_end[1]*1.1,
    f"w* = ({w_rec[0]:.3f}, {w_rec[1]:.3f})",
    color="#ffd166",
    fontsize=9,
    fontweight="bold",
    bbox=dict(boxstyle="round,pad=0.2", facecolor="#1c1c2e", alpha=0.7)
)
ax_geo.axhline(0, color="#555", lw=0.8)
ax_geo.axvline(0, color="#555", lw=0.8)
style_ax(ax_geo,
         title="Subgradient Vectors G (active beamlets)\nColoured by KKT satisfaction",
         xlabel="g₁  (PTV subgradient projection)",
         ylabel="g₂  (OAR subgradient projection)")
ax_geo.legend(facecolor="#1c1c2e", labelcolor="white", fontsize=8)

# Right — per-beamlet KKT residual histogram
ax_hist = axes2[1]
kkt_per_beamlet = np.maximum(G @ w_rec, 0)
ax_hist.hist(kkt_per_beamlet[active],  bins=60, color="#00c8ff", alpha=0.75,
             label=f"Active (n={n_act})",    density=True, zorder=3)
ax_hist.hist(kkt_per_beamlet[~active], bins=60, color="#aaaaaa", alpha=0.50,
             label=f"Inactive (n={n_beamlets-n_act})", density=True, zorder=2)
ax_hist.axvline(0, color="#ff6b6b", ls="--", lw=1.5, label="Zero (ideal)")
style_ax(ax_hist,
         title="Per-beamlet KKT Residual Distribution\n(mass near 0 → stationarity holds)",
         xlabel="max(G w*, 0)  per beamlet",
         ylabel="Density")
ax_hist.legend(facecolor="#1c1c2e", labelcolor="white", fontsize=9)
ax_hist.text(0.97, 0.95,
             f"Overall ‖residual‖\nAll:    {kkt_all:.6f}\nActive: {kkt_active:.6f}",
             transform=ax_hist.transAxes, ha="right", va="top",
             color="white", fontsize=9,
             bbox=dict(boxstyle="round,pad=0.4", facecolor="#1c1c2e", alpha=0.8))

plt.tight_layout(pad=2.0)
plt.subplots_adjust(top=0.90, hspace=0.35, wspace=0.30)
save_and_show(fig2, "fig2_kkt_geometry.png")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 3 — Lambda Sweep (2×2)
# ─────────────────────────────────────────────────────────────────────────────
print("\n── Generating Figure 3: Lambda Sweep ──")
fig3, axes3 = plt.subplots(2, 2, figsize=(12, 9), facecolor=DARK)
fig3.suptitle("IO Phase 3: Effect of Spectral Regularisation (λ sweep)\nRecovered weights w* fixed; varying Frobenius+Group penalty",
              color="white", fontsize=12, fontweight="bold")

sweep_data = [
    (f1v, "F₁  PTV Underdose Penalty",  "#00c8ff", "Lower = better coverage"),
    (f2v, "F₂  OAR Overdose Penalty",   "#ff6b6b", "Lower = better OAR sparing"),
    (nv,  "Nuclear Norm  ‖W‖*",         "#a8ff78", "Lower = more deliverable plan"),
    (rv,  "Effective Rank",              "#ffd166", "Approximate active dimensions"),
]
for idx, (yv, title, col, note) in enumerate(sweep_data):
    ax_ = axes3[idx//2][idx%2]
    ax_.plot(lv, yv, "o-", color=col, lw=2.2, ms=9, zorder=3)
    for xi,(lx,ly) in enumerate(zip(lv,yv)):
        ax_.text(lx, ly, f" {ly:.4f}" if ly<10 else f" {ly:.1f}",
                 color="white", fontsize=8, va="bottom")
    # Shade the "better" direction
    if "Underdose" in title or "Overdose" in title or "Nuclear" in title or "Rank" in title:
        ax_.axhline(min(yv), color=col, ls=":", lw=0.8, alpha=0.4)
    style_ax(ax_, title=title, xlabel="λ (regularisation weight)", ylabel=title.split("  ")[0])
    ax_.set_xticks(lv)
    ax_.text(0.97, 0.92, note, transform=ax_.transAxes,
             ha="right", va="top", color="#aaaaaa", fontsize=8,
             bbox=dict(boxstyle="round,pad=0.3", facecolor="#1c1c2e", alpha=0.7))

plt.tight_layout(pad=2.0)
plt.subplots_adjust(top=0.90, hspace=0.35, wspace=0.30)
save_and_show(fig3, "fig3_lambda_sweep.png")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 4 — DVH: Reference vs Best IO Re-plan (side by side)
# ─────────────────────────────────────────────────────────────────────────────
print("\n── Generating Figure 4: DVH Comparison ──")

best_io_key = "IO λ=0.003 (Fro+Group)" if "IO λ=0.003 (Fro+Group)" in all_plans else labels[-1]
ref_key     = labels[0]
x_best      = all_plans[best_io_key]
x_ref_plot  = all_plans[ref_key]

fig4, (ax_ref, ax_io) = plt.subplots(1, 2, figsize=(14, 6), facecolor=DARK,
                                      sharey=True)
fig4.suptitle("Dose Volume Histograms: Reference Plan vs IO Re-plan (λ=0.003)",
              color="white", fontsize=13, fontweight="bold", y=1.02)

for ax_, xp, title in [(ax_ref, x_ref_plot, f"Reference Plan\n({ref_key})"),
                        (ax_io,  x_best,     f"IO Re-plan\n({best_io_key})")]:
    for sn, Am, lm in strs:
        d   = dose(Am, xp)
        dvh = [(d>=b).mean()*100 for b in bins]
        ax_.plot(bins, dvh, color=SC[sn], lw=2.2, label=sn)
        ax_.axvline(lm, color=SC[sn], lw=0.6, ls=":", alpha=0.4)
        # Annotate D95 for PTV
        if sn == "PTV":
            d95 = float(np.percentile(d,5))
            d05 = float(np.percentile(d,95))
            ax_.axvline(d95, color="#00c8ff", lw=1.0, ls="--", alpha=0.7)
            ax_.text(d95+0.5, 96, f"D95={d95:.1f} Gy", color="#00c8ff", fontsize=8)
            ax_.text(d05-0.5, 7,  f"D05={d05:.1f} Gy", color="#00c8ff", fontsize=8, ha="right")
    style_ax(ax_, title=title,
             xlabel="Dose (Gy)", ylabel="Volume (%)")
    ax_.set_xlim(0, 80); ax_.set_ylim(0, 105)
    ax_.legend(facecolor="#1c1c2e", labelcolor="white", fontsize=10, loc="upper right")

# Annotate the improvement
fig4.text(0.5, -0.04,
          f"Key result: D05 hotspot reduced  {float(df.loc[ref_key,'D05_ptv']):.1f} Gy  →  "
          f"{float(df.loc[best_io_key,'D05_ptv']):.1f} Gy   |   "
          f"D95 coverage maintained  {float(df.loc[ref_key,'D95_ptv']):.1f} → "
          f"{float(df.loc[best_io_key,'D95_ptv']):.1f} Gy",
          ha="center", color="#ffd166", fontsize=10,
          bbox=dict(boxstyle="round,pad=0.4", facecolor="#1c1c2e", alpha=0.8))

plt.tight_layout(pad=2.0)
plt.subplots_adjust(top=0.90, hspace=0.35, wspace=0.30)
save_and_show(fig4, "fig4_dvh_comparison.png")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 5 — Plan Structure: NucNorm vs MU + Beam-wise Intensity
# ─────────────────────────────────────────────────────────────────────────────
print("\n── Generating Figure 5: Plan Structure ──")
fig5, (ax_sc, ax_bm) = plt.subplots(1, 2, figsize=(13, 5.5), facecolor=DARK)
fig5.suptitle("IO Phase 3: Plan Deliverability — Complexity vs Monitor Units",
              color="white", fontsize=13, fontweight="bold", y=1.02)

# Left — NucNorm vs MU scatter
mus  = [float(df.loc[lbl,"total_MU"])  for lbl in labels]
nucs = [float(df.loc[lbl,"nuc_norm"])  for lbl in labels]
for i,(lbl,mu,nuc) in enumerate(zip(labels,mus,nucs)):
    ax_sc.scatter(mu, nuc, s=160, color=PAL[i%4], zorder=4,
                  edgecolors="white", linewidths=0.8)
    ax_sc.annotate(lbl[:20], (mu,nuc),
                   textcoords="offset points", xytext=(8,6),
                   color="white", fontsize=8,
                   bbox=dict(boxstyle="round,pad=0.2", facecolor="#1c1c2e", alpha=0.7))
# Arrow from ref to best IO
if len(mus) >= 2:
    ax_sc.annotate("", xy=(mus[-1],nucs[-1]), xytext=(mus[0],nucs[0]),
                   arrowprops=dict(arrowstyle="->", color="#ffd166", lw=1.8))
    ax_sc.text((mus[0]+mus[-1])/2, (nucs[0]+nucs[-1])/2 + 150,
               "Regularisation\nimproves deliverability",
               ha="center", color="#ffd166", fontsize=9)
style_ax(ax_sc,
         title="Nuclear Norm vs Total Monitor Units\n(bottom-left = simpler, more deliverable)",
         xlabel="Total Monitor Units (MU)",
         ylabel="Nuclear Norm  ‖W‖*")

# Right — beam-wise max intensity
starts = np.cumsum([0]+plan_bpb[:-1])
beam_x = np.arange(nb)
width  = 0.18
for i,(lbl,xp) in enumerate(all_plans.items()):
    bmax = [np.maximum(xp[s:s+sz],0).max() for s,sz in zip(starts,plan_bpb)]
    ax_bm.bar(beam_x + (i - len(all_plans)/2 + 0.5)*width, bmax, width,
              label=lbl[:22], color=PAL[i%4], alpha=0.85, zorder=3)
ax_bm.set_xticks(beam_x)
ax_bm.set_xticklabels([f"Beam {i+1}" for i in range(nb)], color="white", fontsize=9)
style_ax(ax_bm,
         title="Beam-wise Maximum Beamlet Intensity\n(lower & more uniform = better deliverability)",
         xlabel="Beam angle",
         ylabel="Max MU per beamlet")
ax_bm.legend(facecolor="#1c1c2e", labelcolor="white", fontsize=8, loc="upper right")

plt.tight_layout(pad=2.0)
plt.subplots_adjust(top=0.90, hspace=0.35, wspace=0.30)
save_and_show(fig5, "fig5_plan_structure.png")

# ══════════════════════════════════════════════════════════════════════════════
# 10. PROTOCOL PASS / FAIL TABLE
# ══════════════════════════════════════════════════════════════════════════════
lims={"D95_ptv":(">=",57.0,"PTV coverage"),"HI":("<=",0.10,"PTV homogeneity"),
      "Dmax_esoph":("<=",45.0,"Esoph Dmax"),"Dmax_cord":("<=",30.0,"Cord Dmax"),
      "Dmean_lung":("<=",20.0,"Lung Dmean")}
cw=max(26,max(len(l) for l in df.index)+2)
hdr2=f"{'METRIC':<22} {'LIMIT':<8}"+"".join(f"{l:>{cw}}" for l in df.index)
print("\n"+"═"*len(hdr2)+"\n  PROTOCOL PASS / FAIL\n"+"═"*len(hdr2))
print(hdr2+"\n"+"─"*len(hdr2))
for m,(op,lim,desc) in lims.items():
    vals=df[m].astype(float)
    flag=lambda v:("✓PASS" if (v>=lim if op==">=" else v<=lim) else "✗FAIL")
    print(f"{desc:<22} {op}{lim:<7}"+"".join(f"{str(vals[l])+' '+flag(vals[l]):>{cw}}" for l in df.index))
print("═"*len(hdr2))

# ══════════════════════════════════════════════════════════════════════════════
# 11. FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
print("\n"+"═"*70+"\n  FINAL SUMMARY\n"+"═"*70)
print(f"  Reference : {ref_source}")
print(f"  IO method : KKT subgradient QCQP")
print(f"  w* = (w₁={w_rec[0]:.4f}, w₂={w_rec[1]:.4f})")
print(f"  KKT residual (active) = {kkt_active:.6f}  (all) = {kkt_all:.6f}")
print(f"\n  Figures saved to {OUTPUT_DIR}:")
for fn in ["fig1_weight_recovery_kkt.png","fig2_kkt_geometry.png",
           "fig3_lambda_sweep.png","fig4_dvh_comparison.png","fig5_plan_structure.png"]:
    print(f"    {fn}")
print("═"*70)